# Reproduction — *An analysis of the noise schedule for score-based generative models*

Strasman, Ocello, Boyer, Le Corff, Lemaire — TMLR 2025 ([arXiv:2402.04650](https://arxiv.org/abs/2402.04650), PDF dans `references/original_paper.pdf`).

Ce notebook s'appuie sur le package `noise_diff` (`src/noise_diff/`, code officiel des auteurs adapté pour tourner sur CPU, CUDA et MPS) et reproduit :

| Section du notebook | Article |
|---|---|
| 1. Noise schedules | Figures 1, 6, 7, 8, 9 |
| 2. Cas gaussien : bornes KL et $\mathcal W_2$ | Figures 2, 10, 12, 14, 15 · Tables 1–2 · Annexe F |
| 3. Schedule optimal selon la dimension | Figures 3, 13 · Table 2 |
| 4. Cibles non gaussiennes (Funnel, mélange de 25 gaussiennes) | Figures 4, 17, 18 · Table 3 |

Non reproduit : l'expérience CIFAR-10 (figures 5 et 19), qui nécessite les modèles pré-entraînés de Karras et al. (2022).

**Device** : le réseau, l'entraînement et le sampling tournent automatiquement sur le meilleur accélérateur disponible (CUDA > MPS > CPU). Les bornes et les métriques gaussiennes (KL, $\mathcal W_2$) sont toujours calculées sur CPU en float64 : ce sont de petites opérations d'algèbre linéaire, et MPS ne gère ni le float64 ni les valeurs propres.

**Coût** : l'article entraîne un réseau par valeur de $a$ et par répétition. Le preset `"paper"` reprend exactement leurs paramètres (plusieurs heures) ; `"quick"` montre les mêmes tendances en dimension réduite ; `"smoke"` vérifie seulement que tout s'exécute.

In [ ]:
import math
import sys
import time
from pathlib import Path

# Racine du dépôt : premier dossier parent qui contient src/noise_diff
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "noise_diff").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from noise_diff import decoder
from noise_diff import diffusion as diff
from noise_diff import functions as func
from noise_diff import sampler as sp
from noise_diff import synthetic_data as synt
from noise_diff import upperbounds as up
from noise_diff.device import describe_device, get_device, seed_everything

plt.rcParams.update(
    {"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 11}
)

## 0. Configuration

Tout se règle ici. `DEVICE = None` choisit automatiquement le device ; on peut le forcer avec `"cpu"`, `"cuda"` ou `"mps"` (ou la variable d'environnement `NOISE_DIFF_DEVICE`).

Avec des batchs de 64 et un petit réseau, l'entraînement est dominé par le coût de lancement des opérations : le CPU peut alors être aussi rapide que le GPU. Le GPU est nettement plus rapide pour le sampling (10 000 particules).

In [ ]:
DEVICE = None
device = get_device(DEVICE)
print("Device :", describe_device(device))

PRESET = "quick"  # "smoke" (~2 min), "quick" (~45 min), "paper" (paramètres de l'article, plusieurs heures)

PRESETS = {
    "smoke": {
        "d": 5,
        "n_train": 2_000,
        "n_gen": 2_000,
        "a_values": [-5.0, 0.0, 5.0],
        "n_runs": 1,
        "epoch_scale": 0.1,
        "num_steps": 50,
        "num_mc": 100,
        "a_star_grid": np.linspace(-10, 10, 5),
        "dims": [5],
        "dim_study_covariances": ["isotropic"],
        "general_d": 5,
        "general_a_values": [-5.0, 0.0, 5.0],
        "general_epochs": 2,
        "knn_n": 500,
    },
    "quick": {
        "d": 10,
        "n_train": 10_000,
        "n_gen": 10_000,
        "a_values": np.linspace(-10, 10, 11).tolist(),
        "n_runs": 2,
        "epoch_scale": 1.0,
        "num_steps": 100,
        "num_mc": 500,
        "a_star_grid": np.linspace(-10, 10, 21),
        "dims": [5, 10],
        "dim_study_covariances": ["isotropic", "correlated"],
        "general_d": 10,
        "general_a_values": np.linspace(-10, 20, 11).tolist(),
        "general_epochs": 30,
        "knn_n": 2_000,
    },
    "paper": {
        "d": 50,
        "n_train": 10_000,
        "n_gen": 10_000,
        "a_values": np.linspace(-10, 10, 21).tolist(),
        "n_runs": 10,
        "epoch_scale": 1.0,
        "num_steps": 500,
        "num_mc": 500,
        "a_star_grid": np.linspace(-10, 10, 21),
        "dims": [5, 10, 25, 50],
        "dim_study_covariances": ["isotropic", "heteroscedastic", "correlated"],
        "general_d": 50,
        "general_a_values": np.linspace(-10, 20, 31).tolist(),
        "general_epochs": 200,
        "knn_n": 10_000,
    },
}
cfg = PRESETS[PRESET]

# Diffusion (section 5) : T = 1, beta(0) = 0.1, beta(T) = 20, loi stationnaire N(0, sigma^2 Id) avec sigma = 1
T, BETA_MIN, BETA_MAX, SIGMA_INF = 1.0, 0.1, 20.0, 1.0
COSINE_S = 0.021122  # décalage s du schedule cosinus, pour que beta_cos(0) ≈ 0.1 (annexe E.1.3)

# Réseau de la figure 11, Adam, batchs de 64
MID_FEATURES, NUM_LAYERS, BATCH_SIZE = 256, 3, 64
# (epochs, learning rate) par dimension, repris du notebook officiel ; epoch_scale les réduit pour les presets rapides
TRAINING_BY_DIM = {5: (20, 1e-4), 10: (30, 1e-4), 25: (75, 1e-3), 50: (150, 1e-3)}

# Section 2
COVARIANCE = "isotropic"  # "isotropic", "heteroscedastic" ou "correlated"
RESCALE = False  # prétraitement de l'annexe E.2.2, à activer pour heteroscedastic / correlated (figure 15)
LOSS = "explicit"  # "explicit" (section 5.1) ou "conditional" (annexe F)
SCHEMES = ["euler"]  # ajouter "ei" pour l'Exponential Integrator (figure 10)

# Section 4
GENERAL_TARGET = "funnel"  # "funnel" (figures 4 et 17) ou "mg25" (figure 18)

RUN_GAUSSIAN, RUN_DIMENSION_STUDY, RUN_GENERAL = True, True, True
SAVE_OUTPUTS = True  # figures en PDF dans figures/, résultats en CSV dans experiments/
SEED = 20

In [ ]:
FIG_DIR, RESULTS_DIR = ROOT / "figures", ROOT / "experiments"


def save(obj, name):
    if not SAVE_OUTPUTS:
        return
    if isinstance(obj, pd.DataFrame):
        RESULTS_DIR.mkdir(exist_ok=True)
        obj.to_csv(RESULTS_DIR / f"{name}_{PRESET}.csv", index=False)
    else:
        FIG_DIR.mkdir(exist_ok=True)
        obj.savefig(FIG_DIR / f"{name}_{PRESET}.pdf", bbox_inches="tight")


def training_params(d):
    epochs, lr = TRAINING_BY_DIM[min(TRAINING_BY_DIM, key=lambda k: abs(k - d))]
    return max(1, round(epochs * cfg["epoch_scale"])), lr


def make_sde(schedule, d, device):
    # schedule : valeur de a pour la famille paramétrique (a = 0 : linéaire, VPSDE) ou "cosine"
    if schedule == "cosine":
        beta = diff.beta_cosine(T, COSINE_S)
    else:
        beta = diff.beta_parametric(schedule, T, BETA_MIN, BETA_MAX)
    return diff.forward_VPSDE(d, beta, SIGMA_INF, T, device=device)


def train_score(sde, train_data, epochs, lr, exact_score=None):
    # perte explicite si le vrai score est fourni (cas gaussien), sinon perte conditionnelle (denoising)
    net = decoder.Decoder(sde.d, MID_FEATURES, NUM_LAYERS).to(sde.device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    if exact_score is not None:
        loss = diff.loss_explicit(net, sde, exact_score)
    else:
        loss = diff.loss_conditional(net, sde)
    diff.train(
        loss,
        diff.tensor_loader(train_data, BATCH_SIZE),
        epochs,
        optimizer,
        verbose=False,
    )
    return net


SAMPLERS = {"euler": sp.Euler_Maruyama_discr_sampler, "ei": sp.EI_discr_sampler}


def generate(sde, score, n, scheme="euler"):
    # backward discrétisé en num_steps pas, initialisé sous la loi stationnaire ; renvoie un tenseur CPU
    init = sde.final.generate_sample(n)
    return SAMPLERS[scheme](init, sde, score, cfg["num_steps"]).cpu()


def on_device(target, device):
    mu, cov = target.mean_covar()
    return up.gaussian(target.d, mu.to(device), cov.to(device))


def mean_std(values):
    return f"{np.mean(values):.3f} ± {np.std(values):.3f}"

## 1. Noise schedules (figures 1, 6, 7, 8 et 9)

Processus forward (1) : $d\overrightarrow{X}_t = -\frac{\beta(t)}{2\sigma^2}\overrightarrow{X}_t\,dt + \sqrt{\beta(t)}\,dB_t$, simulé exactement par $\overrightarrow{X}_t = m_t X_0 + \sigma_t Z$ avec $m_t = \exp\{-\int_0^t \beta(s)ds / 2\sigma^2\}$ et $\sigma_t^2 = \sigma^2(1 - m_t^2)$.

Famille paramétrique (9) : $\beta_a(t) \propto (e^{at} - 1)/(e^{aT} - 1)$, calibrée pour que $\beta_a(0) = 0.1$ et $\beta_a(1) = 20$ ; $a = 0$ donne le schedule linéaire du VPSDE (Song et al., 2021).

In [ ]:
a_grid = np.linspace(-10, 10, 21)
tt = torch.linspace(0, T, 1000)
colors = plt.cm.coolwarm(np.linspace(0, 1, len(a_grid)))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
for color, a in zip(colors, a_grid, strict=True):
    sde = make_sde(a, 1, "cpu")
    style = (
        {"color": "black", "ls": "--", "lw": 2}
        if a == 0
        else {"color": color, "lw": 1.2}
    )
    axes[0].plot(tt, sde.beta(tt), **style)
    axes[1].plot(tt, sde.mu(tt), **style)
    axes[2].plot(tt, sde.sigma(tt), **style)
for ax, title in zip(
    axes,
    [r"$\beta_a(t)$ (fig. 1)", r"$m_t$ (fig. 7)", r"$\sigma_t$ (fig. 7)"],
    strict=True,
):
    ax.set_title(title)
    ax.set_xlabel("temps de diffusion $t$")
fig.colorbar(
    plt.cm.ScalarMappable(cmap="coolwarm", norm=plt.Normalize(-10, 10)),
    ax=axes,
    label="a (pointillés : a = 0)",
)
save(fig, "fig01_07_parametric_schedules")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
for name, schedule in [("linéaire (a = 0)", 0.0), ("cosinus", "cosine")]:
    sde = make_sde(schedule, 1, "cpu")
    axes[0].plot(tt, torch.clamp(sde.beta(tt), max=200), label=name)
    axes[1].plot(tt, sde.mu(tt), label=name)
    axes[2].plot(tt, sde.sigma(tt), label=name)
for ax, title in zip(
    axes,
    [r"$\beta(t)$, coupé à 200 (fig. 8)", r"$m_t$ (fig. 9)", r"$\sigma_t$ (fig. 9)"],
    strict=True,
):
    ax.set_title(title)
    ax.set_xlabel("temps de diffusion $t$")
    ax.legend()
save(fig, "fig08_09_cosine_vs_linear")
plt.show()

## 2. Cas gaussien : bornes KL et $\mathcal W_2$ (section 5.1, annexes E.2.1, E.2.2 et F)

Cible $\pi_{\rm data} = \mathcal N(\mathbf 1_d, \Sigma)$ avec trois structures de covariance :
- **isotropic** : $\Sigma = 0.5\,I_d$ ;
- **heteroscedastic** : $\Sigma$ diagonale, $\Sigma_{jj} = 1$ pour $j \le 5$ et $0.01$ sinon ;
- **correlated** : $\Sigma_{jj'} = 1/\sqrt{|j - j'| + 1}$ (formule du code officiel ; celle imprimée dans l'article, $1/\sqrt{|j-j'|}$, ne donne pas une matrice définie positive).

Pour chaque $a$ :
1. on entraîne `n_runs` réseaux avec la perte explicite (4), le vrai score étant connu (lemme E.1) ;
2. on génère `n_gen` points avec le schéma d'Euler-Maruyama (50) ;
3. on mesure $\mathrm{KL}(\pi_{\rm data}\,\|\,\hat\pi)$ et $\mathcal W_2(\pi_{\rm data}, \hat\pi)$ par les formules gaussiennes fermées (lemmes E.2 et E.3), à partir de la moyenne et de la covariance empiriques des points générés ;
4. on évalue les bornes :
   - théorème 3.1 : $E_1^{KL} + E_2^{KL} + E_3^{KL}$, où $E_2$ est estimé par Monte-Carlo avec `num_mc` points par pas ;
   - théorème 4.2 : $\varepsilon$ y est estimé par l'erreur $L^2$ maximale du score appris.

Les pointillés des figures correspondent au même SGM avec le **score exact** à la place du réseau.

**Remise à l'échelle** (`RESCALE = True`, annexe E.2.2) : on entraîne sur $X_{\rm scale} = \kappa D^{-1}(X_0 - \mu)$, avec $\kappa = (2\lambda_{\max}(\Sigma^{\rm stand}))^{-1/2}$. On a alors $\lambda_{\max} \le 0.5 < \sigma^2$, comme l'exige le lemme 4.1. La cible devient $\mathcal N(0, \kappa^2 D^{-1}\Sigma D^{-1})$ et les métriques sont calculées dans cet espace. La KL est invariante par cette transformation ; la $\mathcal W_2$ n'est modifiée qu'à une constante près (équation (51)).

In [ ]:
def gaussian_target(kind, d, rescale):
    mu = torch.ones(d)
    if kind == "isotropic":
        cov = 0.5 * torch.eye(d)
    elif kind == "heteroscedastic":
        cov = torch.diag(torch.where(torch.arange(d) < 5, 1.0, 0.01))
    elif kind == "correlated":
        idx = torch.arange(d)
        cov = 1.0 / torch.sqrt((idx[:, None] - idx[None, :]).abs() + 1.0)
    else:
        raise ValueError(kind)
    if rescale:
        inv_std = 1.0 / torch.sqrt(torch.diag(cov))
        cov_stand = inv_std[:, None] * cov * inv_std[None, :]
        kappa = 1.0 / math.sqrt(
            2 * torch.linalg.eigvalsh(cov_stand.double()).max().item()
        )
        mu, cov = torch.zeros(d), kappa**2 * cov_stand
    return up.gaussian(d, mu, cov)


def gaussian_metrics(target, sample):
    # un sampling qui diverge (inf / nan) est enregistré en NaN au lieu d'interrompre l'expérience
    if not torch.isfinite(sample).all():
        return {"kl": np.nan, "w2": np.nan}
    generated = func.empirical(sample)
    return {"kl": func.kl(target, generated), "w2": func.w2(target, generated)}

In [ ]:
if RUN_GAUSSIAN:
    start = time.perf_counter()
    seed_everything(SEED)
    d = cfg["d"]
    epochs, lr = training_params(d)
    target = gaussian_target(COVARIANCE, d, RESCALE)  # CPU : bornes et métriques
    target_dev = on_device(target, device)  # device : entraînement et sampling
    train_data = target_dev.generate_sample(cfg["n_train"])
    train_cpu = train_data.cpu()
    print(
        f"d = {d}, covariance = {COVARIANCE}, rescale = {RESCALE}, loss = {LOSS}, {epochs} epochs, lr = {lr}"
    )

    gaussian_rows, bound_rows = [], []
    for a in tqdm(cfg["a_values"], desc="valeurs de a"):
        sde, sde_cpu = make_sde(a, d, device), make_sde(a, d, "cpu")
        exact_score = diff.explicit_score(sde, target_dev)
        for scheme in SCHEMES:
            sample = generate(sde, exact_score, cfg["n_gen"], scheme)
            gaussian_rows.append(
                dict(
                    a=a,
                    run=-1,
                    score="exact",
                    scheme=scheme,
                    **gaussian_metrics(target, sample),
                )
            )

        e2_values, eps_values = [], []
        for run in range(cfg["n_runs"]):
            net = train_score(
                sde, train_data, epochs, lr, exact_score if LOSS == "explicit" else None
            )
            e2, eps = up.compute_E2(
                target_dev, sde, net, exact_score, cfg["num_steps"], cfg["num_mc"]
            )
            e2_values.append(e2)
            eps_values.append(eps)
            for scheme in SCHEMES:
                sample = generate(sde, net, cfg["n_gen"], scheme)
                gaussian_rows.append(
                    dict(
                        a=a,
                        run=run,
                        score="nn",
                        scheme=scheme,
                        **gaussian_metrics(target, sample),
                    )
                )

        bound_rows.append(
            {
                "a": a,
                "E1": up.compute_E1(target, sde_cpu).item(),
                "E1_no_contraction": up.compute_E1_no_contraction(
                    target, sde_cpu
                ).item(),
                "E2": np.mean(e2_values),
                "E2_std": np.std(e2_values),
                "E3": up.compute_E3(target, sde_cpu, cfg["num_steps"]).item(),
                "epsilon": np.mean(eps_values),
                "W2_bound": up.compute_w2_bound(
                    target, train_cpu, sde_cpu, cfg["num_steps"], np.mean(eps_values)
                ).item(),
            }
        )

    gaussian_df = pd.DataFrame(gaussian_rows)
    bounds_df = pd.DataFrame(bound_rows).set_index("a")
    bounds_df["KL_bound"] = bounds_df[["E1", "E2", "E3"]].sum(axis=1)
    bounds_df["KL_bound_no_contraction"] = bounds_df[
        ["E1_no_contraction", "E2", "E3"]
    ].sum(axis=1)
    tag = f"{COVARIANCE}{'_rescaled' if RESCALE else ''}_d{d}"
    save(gaussian_df, f"gaussian_metrics_{tag}")
    save(bounds_df.reset_index(), f"gaussian_bounds_{tag}")
    print(f"Terminé en {time.perf_counter() - start:.0f} s")
    display(bounds_df.round(4))

In [ ]:
BLUE, ORANGE = "tab:blue", "tab:orange"


def plot_bound_vs_metric(
    ax,
    a_values,
    bound,
    metric_runs,
    metric_exact=None,
    label="",
    bound_band=None,
    extra_bound=None,
    baseline=None,
    baseline_label="VPSDE linéaire (NN)",
):
    # bornes en bleu (axe de gauche), métrique empirique en orange (axe de droite), comme dans l'article
    ax.plot(a_values, bound, color=BLUE, lw=2.5, label="borne supérieure")
    if bound_band is not None:
        ax.fill_between(
            a_values, bound - bound_band, bound + bound_band, color=BLUE, alpha=0.2
        )
    if extra_bound is not None:
        ax.plot(
            a_values,
            extra_bound,
            color=BLUE,
            lw=1.5,
            alpha=0.45,
            label="borne (sans contraction)",
        )
    ax.set_xlabel("a")
    ax.set_ylabel("borne supérieure", color=BLUE)
    ax.tick_params(axis="y", labelcolor=BLUE)
    ax.grid(False)
    ax2 = ax.twinx()
    mean, std = metric_runs.mean(axis=1), metric_runs.std(axis=1).fillna(0)
    ax2.plot(a_values, mean, color=ORANGE, lw=2.5, label=f"{label} (NN)")
    ax2.fill_between(a_values, mean - std, mean + std, color=ORANGE, alpha=0.25)
    if metric_exact is not None:
        ax2.plot(
            a_values,
            metric_exact,
            color=ORANGE,
            ls=":",
            lw=2.5,
            label=f"{label} (score exact)",
        )
    if baseline is not None:
        ax2.axhline(baseline, color=ORANGE, ls="--", lw=1.5, label=baseline_label)
    ax2.set_ylabel(label, color=ORANGE)
    ax2.tick_params(axis="y", labelcolor=ORANGE)
    ax2.grid(False)
    handles = ax.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
    ax.legend(handles=handles, fontsize=8, loc="best")


def per_run(df, metric, scheme):
    nn = df[(df.score == "nn") & (df.scheme == scheme)]
    return nn.pivot(index="a", columns="run", values=metric)


def exact(df, metric, scheme):
    return df[(df.score == "exact") & (df.scheme == scheme)].set_index("a")[metric]

### Figure 2 (et 10 avec `SCHEMES = ["euler", "ei"]`) : métriques empiriques et bornes en fonction de $a$

Le trait plein orange donne la moyenne ± l'écart-type sur les répétitions ; le tireté marque le VPSDE linéaire ($a = 0$). Pour le cas *heteroscedastic* ou *correlated*, l'article ne montre la borne $\mathcal W_2$ qu'après remise à l'échelle (figure 15, `RESCALE = True`) : sans elle, l'hypothèse $\lambda_{\max}(\Sigma) < \sigma^2$ du lemme 4.1 n'est pas satisfaite.

In [ ]:
if RUN_GAUSSIAN:
    a_values = bounds_df.index.values
    for scheme in SCHEMES:
        kl_runs, w2_runs = (
            per_run(gaussian_df, "kl", scheme),
            per_run(gaussian_df, "w2", scheme),
        )
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), layout="constrained")
        plot_bound_vs_metric(
            axes[0],
            a_values,
            bounds_df.KL_bound,
            kl_runs,
            exact(gaussian_df, "kl", scheme),
            label=r"KL($\pi_{\rm data}\|\hat\pi$)",
            bound_band=bounds_df.E2_std,
            extra_bound=bounds_df.KL_bound_no_contraction,
            baseline=kl_runs.mean(axis=1).get(0.0),
        )
        plot_bound_vs_metric(
            axes[1],
            a_values,
            bounds_df.W2_bound,
            w2_runs,
            exact(gaussian_df, "w2", scheme),
            label=r"$\mathcal{W}_2(\pi_{\rm data}, \hat\pi)$",
            baseline=w2_runs.mean(axis=1).get(0.0),
        )
        axes[0].set_title("Théorème 3.1 (KL)")
        axes[1].set_title(r"Théorème 4.2 ($\mathcal{W}_2$)")
        fig.suptitle(
            f"{COVARIANCE}{' (rescaled)' if RESCALE else ''}, d = {d}, schéma {scheme}, perte {LOSS}"
        )
        save(fig, f"fig02_bounds_{tag}_{scheme}_{LOSS}")
        plt.show()

### Figure 12 : décomposition de la borne KL

$E_1$ est l'erreur de mélange (avec et sans l'argument de contraction de la proposition C.1), $E_2$ l'erreur d'approximation du score et $E_3$ l'erreur de discrétisation. $E_3$ ne dépend que de $\beta(T)$, identique pour tous les $a$.

In [ ]:
if RUN_GAUSSIAN:
    fig, ax = plt.subplots(figsize=(7, 4.5), layout="constrained")
    ax.plot(a_values, bounds_df.E1, lw=2, label="$E_1$ mélange (contraction)")
    ax.plot(
        a_values,
        bounds_df.E1_no_contraction,
        lw=2,
        ls="--",
        label="$E_1$ mélange (sans contraction)",
    )
    ax.plot(a_values, bounds_df.E2, lw=2, label="$E_2$ approximation")
    ax.fill_between(
        a_values,
        bounds_df.E2 - bounds_df.E2_std,
        bounds_df.E2 + bounds_df.E2_std,
        alpha=0.2,
    )
    ax.plot(a_values, bounds_df.E3, lw=2, label="$E_3$ discrétisation")
    ax.set_yscale("log")
    ax.set_xlabel("a")
    ax.set_title(f"Termes de la borne KL — {COVARIANCE}, d = {d}")
    ax.legend()
    save(fig, f"fig12_kl_terms_{tag}")
    plt.show()

### Tables 1 et 2 (pour la dimension choisie) : $a^\star$ qui minimise la borne vs schedule linéaire

In [ ]:
if RUN_GAUSSIAN:
    scheme = SCHEMES[0]
    rows = []
    for metric, bound_col in [("kl", "KL_bound"), ("w2", "W2_bound")]:
        runs = per_run(gaussian_df, metric, scheme)
        a_star = bounds_df[bound_col].idxmin()
        rows.append(
            {
                "métrique": metric.upper(),
                "a* (argmin borne)": a_star,
                "a* : métrique": mean_std(runs.loc[a_star]),
                "linéaire (a = 0) : métrique": mean_std(runs.loc[0.0])
                if 0.0 in runs.index
                else "—",
                "meilleur a empirique": runs.mean(axis=1).idxmin(),
            }
        )
    display(pd.DataFrame(rows))

## 3. Schedule optimal selon la dimension (figure 3, table 2)

Pour chaque dimension et chaque covariance, on cherche $a^\star$ qui minimise la borne $\mathcal W_2$ du théorème 4.2. On balaie d'abord la grille `a_star_grid`, puis on affine avec un pas de 0.25 autour du minimum, comme dans l'annexe E.2.2. On compare ensuite la $\mathcal W_2$ obtenue avec $\beta_{a^\star}$, avec le schedule linéaire $\beta_0$ et avec le schedule cosinus $\beta_{\cos}$.

Écart à l'article : pour éviter d'entraîner un réseau par valeur de $a$ avant de choisir $a^\star$, le terme $\varepsilon T\beta(T)$ est omis de la borne, comme dans l'annexe E.2.3. Les covariances *heteroscedastic* et *correlated* sont remises à l'échelle, comme dans la figure 3.

In [ ]:
def w2_bound_without_eps(target, train_cpu, a):
    return up.compute_w2_bound(
        target, train_cpu, make_sde(a, target.d, "cpu"), cfg["num_steps"], 0.0
    ).item()


def find_a_star(target, train_cpu):
    grid = np.asarray(cfg["a_star_grid"])
    coarse = grid[np.argmin([w2_bound_without_eps(target, train_cpu, a) for a in grid])]
    fine = np.arange(coarse - 1, coarse + 1.001, 0.25)
    return float(
        fine[np.argmin([w2_bound_without_eps(target, train_cpu, a) for a in fine])]
    )


if RUN_DIMENSION_STUDY:
    start = time.perf_counter()
    seed_everything(SEED)
    dim_rows = []
    for kind in cfg["dim_study_covariances"]:
        for d_ in cfg["dims"]:
            target_ = gaussian_target(kind, d_, rescale=kind != "isotropic")
            target_dev_ = on_device(target_, device)
            train_ = target_dev_.generate_sample(cfg["n_train"])
            a_star = find_a_star(target_, train_.cpu())
            epochs_, lr_ = training_params(d_)
            for name, schedule in [
                ("a*", a_star),
                ("linéaire", 0.0),
                ("cosinus", "cosine"),
            ]:
                sde_ = make_sde(schedule, d_, device)
                exact_score_ = diff.explicit_score(sde_, target_dev_)
                for run in tqdm(
                    range(cfg["n_runs"]), desc=f"{kind}, d = {d_}, {name}", leave=False
                ):
                    net = train_score(sde_, train_, epochs_, lr_, exact_score_)
                    sample = generate(sde_, net, cfg["n_gen"])
                    dim_rows.append(
                        dict(
                            covariance=kind,
                            d=d_,
                            schedule=name,
                            a_star=a_star,
                            run=run,
                            **gaussian_metrics(target_, sample),
                        )
                    )
    dim_df = pd.DataFrame(dim_rows)
    save(dim_df, "dimension_study")
    print(f"Terminé en {time.perf_counter() - start:.0f} s")

In [ ]:
if RUN_DIMENSION_STUDY:
    kinds = cfg["dim_study_covariances"]
    fig, axes = plt.subplots(
        1,
        len(kinds),
        figsize=(5.5 * len(kinds), 4.2),
        layout="constrained",
        squeeze=False,
    )
    styles = {
        "a*": {"color": "tab:blue", "ls": "-"},
        "linéaire": {"color": "goldenrod", "ls": "--"},
        "cosinus": {"color": "tab:orange", "ls": ":"},
    }
    for ax, kind in zip(axes[0], kinds, strict=True):
        sub = dim_df[dim_df.covariance == kind]
        for name, style in styles.items():
            stats = (
                sub[sub.schedule == name].groupby("d").w2.agg(["mean", "std"]).fillna(0)
            )
            ax.plot(stats.index, stats["mean"], marker="o", lw=2, label=name, **style)
            ax.fill_between(
                stats.index,
                stats["mean"] - stats["std"],
                stats["mean"] + stats["std"],
                color=style["color"],
                alpha=0.15,
            )
        ax.set_xticks(cfg["dims"])
        ax.set_xlabel("dimension d")
        ax.set_ylabel(r"$\mathcal{W}_2$")
        ax.set_title(kind + ("" if kind == "isotropic" else " (rescaled)"))
        ax.legend()
    save(fig, "fig03_dimension_study")
    plt.show()

    table = (
        dim_df.groupby(["covariance", "d", "schedule"])
        .w2.apply(lambda s: mean_std(s.values))
        .unstack()
    )
    a_stars = dim_df.groupby(["covariance", "d"]).a_star.first().rename("valeur de a*")
    display(table.join(a_stars))

## 4. Cibles non gaussiennes (section 5.2, annexe E.2.3, figures 4, 17 et 18, table 3)

- **Funnel** : $\pi_{\rm data}(x) = \varphi_{a^2}(x_1)\prod_{j\ge 2}\varphi_{\exp(2bx_1)}(x_j)$ avec $a = 1$, $b = 0.5$.
- **MG25** : mélange de 25 gaussiennes de moyennes $(j, k, 0, \dots, 0)$, $j, k \in \{-2, \dots, 2\}$, et de covariance $\mathrm{diag}(0.01, 0.01, 0.1, \dots, 0.1)$.

Les données sont standardisées puis remises à l'échelle (annexe E.2.2). Le score n'étant plus connu, les réseaux sont entraînés avec la perte conditionnelle (5). Les points générés sont ramenés dans l'espace d'origine, puis comparés à un échantillon test de $\pi_{\rm data}$ selon trois métriques :
- la $\mathcal W_2$ *sliced*, avec 2000 projections ;
- la KL estimée par $k$ plus proches voisins (Wang et al., 2009), avec $k = \lceil\sqrt d\rceil$ ;
- la log-vraisemblance négative, sur 1000 points générés.

La borne du théorème 4.2 est évaluée sur l'approximation gaussienne des données remises à l'échelle, sans le terme $\varepsilon$, comme dans l'article.

In [ ]:
if RUN_GENERAL:
    start = time.perf_counter()
    seed_everything(SEED)
    d = cfg["general_d"]
    distribution = synt.Funnel(d) if GENERAL_TARGET == "funnel" else synt.MG25(d)
    raw, test = distribution.sample(cfg["n_train"]), distribution.sample(cfg["n_gen"])

    # standardisation puis facteur kappa = 1 / sqrt(2 lambda_max) : lambda_max des données d'entraînement = 0.5
    x_stand, means, stds = func.normalize(raw)
    rescale = math.sqrt(
        2
        * torch.linalg.eigvalsh(func.empirical_mean_covar(x_stand)[1].double())
        .max()
        .item()
    )
    train_scaled = x_stand / rescale
    train_dev = train_scaled.to(device)

    epochs, lr = cfg["general_epochs"], training_params(d)[1]
    k_nn, n_knn = math.ceil(math.sqrt(d)), cfg["knn_n"]
    general_rows, general_bounds = [], []
    for schedule in tqdm([*cfg["general_a_values"], "cosine"], desc="schedules"):
        sde = make_sde(schedule, d, device)
        if schedule != "cosine":
            sde_cpu = make_sde(schedule, d, "cpu")
            bound = up.compute_w2_bound(
                None, train_scaled, sde_cpu, cfg["num_steps"], 0.0, gauss=False
            )
            general_bounds.append({"a": schedule, "W2_bound": bound.item()})
        for run in range(cfg["n_runs"]):
            net = train_score(sde, train_dev, epochs, lr)
            generated = func.unnormalize(
                generate(sde, net, cfg["n_gen"]), means, stds, rescale
            )
            n_diverged = (~torch.isfinite(generated).all(dim=1)).sum().item()
            if n_diverged:
                general_rows.append(
                    {
                        "schedule": schedule,
                        "run": run,
                        "diverged": n_diverged,
                        "sliced_w2": np.nan,
                        "knn_kl": np.nan,
                        "nll": np.nan,
                    }
                )
                continue
            general_rows.append(
                {
                    "schedule": schedule,
                    "run": run,
                    "diverged": 0,
                    "sliced_w2": func.sliced_w2(
                        test, generated, n_projections=2000, seed=run
                    ),
                    "knn_kl": func.knn_estimator_torch(
                        test[:n_knn], generated[:n_knn], k_nn
                    ).item(),
                    "nll": distribution.compute_NLL(generated[:1000].numpy()),
                }
            )
    general_df = pd.DataFrame(general_rows)
    general_bounds_df = pd.DataFrame(general_bounds).set_index("a")
    diverged = general_df[general_df.diverged > 0]
    if len(diverged):
        print(
            "Attention, sampling divergent (points non finis, métriques mises à NaN) :"
        )
        display(diverged[["schedule", "run", "diverged"]])
    save(general_df, f"general_metrics_{GENERAL_TARGET}_d{d}")
    save(general_bounds_df.reset_index(), f"general_bounds_{GENERAL_TARGET}_d{d}")
    print(f"Terminé en {time.perf_counter() - start:.0f} s")

In [ ]:
if RUN_GENERAL:
    metrics = [
        ("sliced_w2", r"sliced-$\mathcal{W}_2$"),
        ("knn_kl", "KL (k plus proches voisins)"),
        ("nll", "log-vraisemblance négative"),
    ]
    param = general_df[general_df.schedule != "cosine"].astype({"schedule": float})
    cosine = general_df[general_df.schedule == "cosine"]
    a_values = general_bounds_df.index.values
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.8), layout="constrained")
    for ax, (metric, label) in zip(axes, metrics, strict=True):
        runs = param.pivot(index="schedule", columns="run", values=metric)
        plot_bound_vs_metric(
            ax,
            a_values,
            general_bounds_df.W2_bound,
            runs,
            label=label,
            baseline=cosine[metric].mean(),
            baseline_label="schedule cosinus",
        )
        ax.set_title(label)
    fig.suptitle(
        f"{GENERAL_TARGET}, d = {d} (figures {'4 et 17' if GENERAL_TARGET == 'funnel' else '18'})"
    )
    save(fig, f"fig04_17_18_{GENERAL_TARGET}_d{d}")
    plt.show()

    a_star = general_bounds_df.W2_bound.idxmin()
    selection = {
        "a* (argmin borne)": param[param.schedule == a_star],
        "cosinus": cosine,
    }
    if 0.0 in param.schedule.values:
        selection["linéaire (a = 0)"] = param[param.schedule == 0.0]
    table = pd.DataFrame(
        {
            name: {label: mean_std(rows[metric]) for metric, label in metrics}
            for name, rows in selection.items()
        }
    )
    print(f"a* = {a_star}  (table 3)")
    display(table)

## Récapitulatif des paramètres

In [ ]:
print(f"preset = {PRESET}, device = {describe_device(device)}, seed = {SEED}")
display(
    pd.Series(
        {k: v for k, v in cfg.items() if not isinstance(v, (list, np.ndarray))},
        name="valeur",
        dtype=object,
    ).to_frame()
)